# Reentrenamiento del Modelo Prophet
En este notebook, cargaremos el modelo Prophet previamente guardado, lo actualizaremos con nuevos datos y lo reentrenaremos. Lo vamos a hacer de manera mock, es decir, neustro dataset lo tenemos recortado para simular que recibimos nuevos datos.

In [2]:
# Importar librerías necesarias
import pandas as pd
import joblib
from prophet import Prophet

c:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cargar el Modelo Guardado
Cargaremos el modelo Prophet previamente guardado en un archivo .pkl.

In [6]:
# Cargar el modelo Prophet guardado
modelo_path = 'modelo_prophet.pkl'
m = joblib.load(modelo_path)

## Cargar Nuevos Datos
Vamos a cargar el nuevo conjunto de datos (simulando que son datos nuevos)

In [5]:
#cargar los datos de pasados
datos_pasados = pd.read_csv('train.csv')

#datos nuevos
datos_nuevos = pd.read_csv('test.csv')

#unir los datos pasados y nuevos
datos_completos = pd.concat([datos_pasados, datos_nuevos], ignore_index=True)

In [7]:
import pandas as pd
import holidays

def crear_regresores(df: pd.DataFrame,
                     festivos_region: str = "CT",
                     festivos_pais: str = "Spain") -> pd.DataFrame:
    
    df = df.copy()
    df["ds"] = pd.to_datetime(df["ds"])

    df["y_lag_1"]  = df["y"].shift(1)
    df["y_lag_2"]  = df["y"].shift(2)
    df["y_lag_168"] = df["y"].shift(168)     
    df["y_lag_24"] = df["y"].shift(24)
    df["y_lag_48"] = df["y"].shift(48)
    df["y_lag_72"] = df["y"].shift(72)

    df["coste_euros_lag_1"] = df["coste_euros"].shift(1)
    df["coste_euros_lag_2"] = df["coste_euros"].shift(2)
    df["coste_euros_lag_24"] = df["coste_euros"].shift(24)
    df["coste_euros_lag_48"] = df["coste_euros"].shift(48)

    df["y_moving_avg_3"] = df["y"].shift(1).rolling(3).mean()
    df["y_moving_avg_6"] = df["y"].shift(1).rolling(6).mean()
    df["y_moving_avg_24"] = df["y"].shift(1).rolling(24).mean()

    df["coste_euros_moving_avg_3"] = df["coste_euros"].shift(1).rolling(3).mean()
    df["coste_euros_moving_avg_6"] = df["coste_euros"].shift(1).rolling(6).mean()
    df["coste_euros_moving_avg_24"] = df["coste_euros"].shift(1).rolling(24).mean()

    df["week_avg"] = df["y"].shift(1).rolling(168).mean()
    df["trend_diff"] = df["y"].shift(1) - df["week_avg"]

    df["dia_semana"] = df["ds"].dt.dayofweek             
    df["es_finde"]   = (df["dia_semana"] >= 5).astype(int)

    años = range(df["ds"].dt.year.min(), df["ds"].dt.year.max() + 1)
    festivos = holidays.country_holidays(festivos_pais, years=años,
                                         subdiv=festivos_region)
    df["es_festivo"] = df["ds"].dt.date.isin(festivos).astype(int)
    df["year"] = df["ds"].dt.year
    df["month"] = df["ds"].dt.month

    df.dropna(inplace=True)

    return df

In [8]:
# hacer regresores
datos_completos = crear_regresores(datos_completos)

## Reentrenar el Modelo
Actualizaremos el modelo Prophet con los nuevos datos.

In [12]:
# Replica los mismos argumentos y regresores que usaste la 1.ª vez
m_nuevo = Prophet(
    changepoint_prior_scale = m.params['delta'].shape[0] and m.changepoint_prior_scale or 0.05,
    yearly_seasonality      = m.yearly_seasonality,
    weekly_seasonality      = m.weekly_seasonality,
    daily_seasonality       = m.daily_seasonality,
)
for name, params in m.extra_regressors.items():
    m_nuevo.add_regressor(name, mode=params['mode'])

# Re‑ajuste con el set completo y los parámetros antiguos como inicio
m_nuevo.fit(datos_completos, init=m.params)    

19:12:41 - cmdstanpy - INFO - Chain [1] start processing
19:12:54 - cmdstanpy - INFO - Chain [1] done processing


## Guardar el Modelo Reentrenado
Guardaremos el modelo actualizado en un archivo .pkl.

In [ ]:
# Guardar el modelo reentrenado
modelo_reentrenado_path = 'modelo_prophet_reentrenado.pkl'
joblib.dump(m_nuevo, modelo_reentrenado_path)